# 02 — Data Splitting & Preprocessing (Phase 2)

Wraps `src/data/split.py` and the preprocessing pipeline builder. Key
invariant demonstrated here: **the test set is split off first and is not
touched again until final evaluation** — every transformer below is fit on
`X_train` only.

In [ ]:
from src.config import get_default_config
from src.data.load import load_and_validate_data
from src.data.split import train_test_split_data
from src.data.preprocessing import build_preprocessor  # ColumnTransformer factory

config = get_default_config()
config.data.target_column = "<set explicitly — dataset-specific>"

df, schema = load_and_validate_data(config.data)

# Feed discovered schema into preprocessing config (never hardcoded)
config.preprocessing.numerical_features = schema.numerical_features
config.preprocessing.categorical_features = schema.categorical_features

In [ ]:
X_train, X_test, y_train, y_test = train_test_split_data(
    df, config.data.target_column, config.split
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train target balance:\n", y_train.value_counts(normalize=True))
print("Test target balance:\n", y_test.value_counts(normalize=True))

Stratified split (`config.split.stratify=True` by default) should keep train
and test target ratios close — a large discrepancy above would indicate a
bug in the split logic.

## Build and fit the preprocessor (train-only fit)

In [ ]:
preprocessor = build_preprocessor(config.preprocessing)

# Fit strictly on X_train — this is the leakage-prevention boundary.
preprocessor.fit(X_train)

X_train_transformed = preprocessor.transform(X_train)
feature_names_out = preprocessor.get_feature_names_out()

print("Transformed shape:", X_train_transformed.shape)
print("n transformed features:", len(feature_names_out))
feature_names_out[:20]

In [ ]:
# Sanity check: transforming test set with the SAME fitted preprocessor
# (never re-fit on test) should succeed without shape mismatches.
X_test_transformed = preprocessor.transform(X_test)
print("Test transformed shape:", X_test_transformed.shape)
assert X_test_transformed.shape[1] == X_train_transformed.shape[1]

## Unknown category handling check

`handle_unknown_categorical="ignore"` (default) means categories seen only
in test/production data won't crash inference — they'll be encoded as all-zero
one-hot rows. Worth explicitly confirming this behavior once on real data.

In [ ]:
for col in schema.categorical_features:
    train_cats = set(X_train[col].dropna().unique())
    test_cats = set(X_test[col].dropna().unique())
    unseen = test_cats - train_cats
    if unseen:
        print(f"{col}: categories in test not seen in train -> {unseen}")

## Notes

- This notebook fits the preprocessor standalone purely for inspection.
  In actual training (Phase 3+), the preprocessor lives *inside* the model
  `Pipeline` (`build_model_pipeline`), so it is refit correctly per CV fold /
  per calibration split — never fit once globally and reused, which would leak.
- `X_test` / `y_test` are not used for anything beyond the shape/category
  sanity checks above in this notebook.